# Stage B — Fit speaker models, anonymize, and emit data dictionary

Loads the per-participant table produced by `process_data.ipynb`, fits 7
speaker models per scenario per participant (literal + 6 pragmatic
combinations of `psi ∈ {inf, pers+, pers-}` × `update_internal ∈ {T, F}`),
merges the fits back onto the participant rows, drops identifying
columns, and writes:

- `raw_do_not_track/speaker_n1_fitted.csv` — full table including identifiers (do not share)
- `speaker_n1_fitted_anonymized.csv` — public-facing wide table (one row per participant)
- `data_dictionary.csv` — per-column profile of the anonymized table

See [`README.md`](./README.md) for the modeling details and column conventions.


## Imports

In [1]:
import pandas as pd
import numpy as np
import warnings
from pathlib import Path
from typing import List, Dict, Tuple, Any, Optional
from tqdm import tqdm

# Import RSA modules (these should be in the same directory or in PYTHONPATH)
import sys
# Uncomment and modify if needed:
# sys.path.append('/path/to/rsa_modules/')

from rsa_optimal_exp_core import World, LiteralSpeaker, PragmaticSpeaker_obs
from rsa_optimal_exp_fitting import log_likelihood_utt_seq, log_likelihood_alpha_opt_utt_seq

# Suppress warnings for cleaner output
warnings.filterwarnings('ignore')

/Users/kangke/projects/prag_net/data/cogsci_rsa_speaker_experiment_n5/rsa_optimal_exp_core.py:2048: SyntaxWarning: invalid escape sequence '\p'
  Returns P(theta) \propto exp( sum_{psi, alpha} log_joint(theta,psi,alpha) ).
/Users/kangke/projects/prag_net/data/cogsci_rsa_speaker_experiment_n5/rsa_optimal_exp_core.py:2061: SyntaxWarning: invalid escape sequence '\p'
  Returns P(psi) \propto exp( sum_{theta, alpha} log_joint(theta,psi,alpha) ).
/Users/kangke/projects/prag_net/data/cogsci_rsa_speaker_experiment_n5/rsa_optimal_exp_core.py:2071: SyntaxWarning: invalid escape sequence '\p'
  Returns P(alpha) \propto exp( sum_{theta, psi} log_joint(theta,psi,alpha)).


## Configuration

In [2]:
# Experiment configuration
# n=1: One group/experiment
# m=5: 5 patients (Bernoulli trials) per group
N_EXPERIMENTS = 1  # Number of independent experiments
M_PATIENTS = 5     # Number of patients per experiment (Bernoulli trials)

# Model fitting configuration
ALPHA_BOUNDS = (0.001, 100.0)  # Bounds for alpha optimization
GRID_SEARCH = False            # Use grid search (more robust)
GRID_POINTS = 300             # Number of grid points
GRID_SPACING = "log"          # Logarithmic spacing
INCLUDE_DETERM = True         # Include deterministic alpha

# Expected number of trials per condition
EXPECTED_TRIALS = 10

# Mapping from experiment data to RSA model
PREDICATE_MAP = {'Effective': 'successful', 'Ineffective': 'unsuccessful'}
QUANTIFIER_MAP = {'All': 'all', 'Most': 'most', 'Some': 'some', 'No': 'no'}

# Goal conditions in the experiment
GOAL_CONDITIONS = ['inf', 'persp', 'persm']

# All models to fit for each condition
# Format: (model_name, psi, update_internal)
# psi: 'inf', 'pers+', 'pers-'
# update_internal: True (dynamic), False (static)
MODELS_TO_FIT = [
    ('literal', None, None),           # Literal model (no psi/update)
    ('inf_T', 'inf', True),            # Informative, dynamic
    ('inf_F', 'inf', False),           # Informative, static
    ('persp_T', 'pers+', True),        # Persuade+, dynamic
    ('persp_F', 'pers+', False),       # Persuade+, static
    ('persm_T', 'pers-', True),        # Persuade-, dynamic
    ('persm_F', 'pers-', False),       # Persuade-, static
]

MODEL_NAMES = [m[0] for m in MODELS_TO_FIT]

## Helpers — World construction, data ↔ RSA shape conversion, trial extraction

In [3]:
def create_world(n: int = N_EXPERIMENTS, m: int = M_PATIENTS) -> World:
    """Create the RSA World object for the experiment."""
    return World(n=n, m=m)


def num_effective_to_observation(num_effective: int, m: int = M_PATIENTS) -> Tuple[int, ...]:
    """
    Convert num_effective (count of effective patients) to observation tuple.
    
    For n=1, m=5, observation is a one-hot encoding of length m+1=6:
    - num_effective=0 -> (1, 0, 0, 0, 0, 0)
    - num_effective=3 -> (0, 0, 0, 1, 0, 0)
    - num_effective=5 -> (0, 0, 0, 0, 0, 1)
    """
    n_effective = int(num_effective)
    return tuple(1 if i == n_effective else 0 for i in range(m + 1))


def format_utterance(predicate: str, quantifier: str) -> str:
    """
    Convert experiment predicate/quantifier to RSA utterance format.
    
    RSA uses comma-separated format: "quantifier,predicate"
    Example: ('Effective', 'Most') -> 'most,successful'
    """
    pred = PREDICATE_MAP.get(predicate, predicate.lower())
    quant = QUANTIFIER_MAP.get(quantifier, quantifier.lower())
    return f"{quant},{pred}"


def extract_trials_for_condition(df_row: pd.Series, condition: str) -> Dict[str, Any]:
    """
    Extract observation and utterance sequences for a specific condition.
    
    Parameters
    ----------
    df_row : pd.Series
        A row from the processed data (one participant)
    condition : str
        Condition prefix: 'inf', 'persp', or 'persm'
    
    Returns
    -------
    Dict with:
        - obs_seq: List of observation tuples
        - utt_seq: List of utterance strings
        - n_trials: Number of valid trials
        - is_complete: Whether all expected trials are present
    """
    obs_seq = []
    utt_seq = []
    
    for r in range(1, 11):  # Rounds 1-10
        num_eff_col = f'{condition}_r{r}_num_effective'
        pred_col = f'{condition}_r{r}_predicate'
        quant_col = f'{condition}_r{r}_quantifier'
        
        # Check if columns exist and have valid values
        if num_eff_col not in df_row.index:
            continue
            
        num_eff = df_row.get(num_eff_col)
        pred = df_row.get(pred_col)
        quant = df_row.get(quant_col)
        
        # Skip if any value is missing
        if pd.isna(num_eff) or pd.isna(pred) or pd.isna(quant):
            continue
        if pred == '' or quant == '':
            continue
            
        obs = num_effective_to_observation(int(num_eff))
        utt = format_utterance(pred, quant)
        
        obs_seq.append(obs)
        utt_seq.append(utt)
    
    n_trials = len(obs_seq)
    is_complete = (n_trials == EXPECTED_TRIALS)
    
    return {
        'obs_seq': obs_seq,
        'utt_seq': utt_seq,
        'n_trials': n_trials,
        'is_complete': is_complete
    }

## Fit functions

`fit_pragmatic_model` optimizes `alpha` to maximize the utterance-sequence
log-likelihood under the given `(psi, update_internal)` model. With
`INCLUDE_DETERM=True`, the deterministic-argmax limit `alpha="determ"` is
included as a candidate.

In [4]:
def fit_literal_model(
    world: World,
    obs_seq: List[Tuple[int, ...]],
    utt_seq: List[str]
) -> Dict[str, Any]:
    """
    Fit literal speaker model (no alpha optimization needed).
    
    Returns
    -------
    Dict with:
        - ll: float (log-likelihood)
        - alpha: None (literal has no alpha)
    """
    config = {
        'speaker_type': 'literal',
        'initial_beliefs_theta': None
    }
    
    try:
        ll = log_likelihood_utt_seq(world, obs_seq, utt_seq, config)
    except Exception as e:
        ll = np.nan
    
    return {
        'll': ll,
        'alpha': None
    }


def fit_pragmatic_model(
    world: World,
    obs_seq: List[Tuple[int, ...]],
    utt_seq: List[str],
    psi: str,
    update_internal: bool,
    alpha_bounds: Tuple[float, float] = ALPHA_BOUNDS,
    grid_search: bool = GRID_SEARCH,
    grid_points: int = GRID_POINTS,
    grid_spacing: str = GRID_SPACING,
    include_determ: bool = INCLUDE_DETERM
) -> Dict[str, Any]:
    """
    Fit pragmatic speaker model with alpha optimization.
    
    Parameters
    ----------
    world : World
        RSA World object
    obs_seq : List of observation tuples
    utt_seq : List of utterance strings
    psi : str
        Speaker goal: 'inf', 'pers+', or 'pers-'
    update_internal : bool
        True for dynamic model, False for static model
    
    Returns
    -------
    Dict with:
        - ll: float (log-likelihood at optimal alpha)
        - alpha: float or 'determ' or None
    """
    config = {
        'speaker_type': 'pragmatic',
        'omega': 'strat',  # Strategic speaker
        'psi': psi,
        'update_internal': update_internal,
        'beta': 0.0,  # Pure goal (no informativeness mixing)
        'initial_beliefs_theta': None
    }
    
    try:
        result = log_likelihood_alpha_opt_utt_seq(
            world=world,
            obs_seq=obs_seq,
            utt_seq=utt_seq,
            speaker_config=config,
            alpha_bounds=alpha_bounds,
            grid_search=grid_search,
            grid_points=grid_points,
            grid_spacing=grid_spacing,
            include_determ=include_determ
        )
        
        return {
            'll': result['max_log_likelihood'],
            'alpha': result['optimal_alpha']
        }
        
    except Exception as e:
        return {
            'll': np.nan,
            'alpha': None
        }


def fit_all_models_for_sequence(
    world: World,
    obs_seq: List[Tuple[int, ...]],
    utt_seq: List[str]
) -> Dict[str, Dict[str, Any]]:
    """
    Fit all 6 models for one observation-utterance sequence.
    
    Returns
    -------
    Dict with model names as keys, each containing {'ll': float, 'alpha': value}
    """
    results = {}
    
    for model_name, psi, update_internal in MODELS_TO_FIT:
        if model_name == 'literal':
            results[model_name] = fit_literal_model(world, obs_seq, utt_seq)
        else:
            results[model_name] = fit_pragmatic_model(
                world, obs_seq, utt_seq, psi, update_internal
            )
    
    return results

## Per-participant pipeline

In [5]:
def process_participant_wide(
    df_row: pd.Series,
    world: World
) -> Dict[str, Any]:
    """
    Process all conditions for a single participant (wide format).
    
    Returns dict with one entry per condition-model combination.
    """
    participant_id = df_row.get('participant_id', df_row.name)
    result = {'participant_id': participant_id}
    
    for condition in GOAL_CONDITIONS:
        # Extract trial data
        trial_data = extract_trials_for_condition(df_row, condition)
        
        # Store trial info
        result[f'{condition}_n_trials'] = trial_data['n_trials']
        result[f'{condition}_is_complete'] = trial_data['is_complete']
        
        # Fit all models if we have any trials
        if trial_data['n_trials'] > 0:
            model_results = fit_all_models_for_sequence(
                world,
                trial_data['obs_seq'],
                trial_data['utt_seq']
            )
            
            # Store results for each model
            for model_name in MODEL_NAMES:
                mr = model_results[model_name]
                result[f'{condition}_{model_name}_ll'] = mr['ll']
                result[f'{condition}_{model_name}_alpha'] = mr['alpha']
        else:
            # No trials - fill with NaN
            for model_name in MODEL_NAMES:
                result[f'{condition}_{model_name}_ll'] = np.nan
                result[f'{condition}_{model_name}_alpha'] = None
    
    return result


def fit_all_participants_wide(
    df: pd.DataFrame,
    verbose: bool = True
) -> pd.DataFrame:
    """
    Fit all models for all participants (wide format).
    
    Parameters
    ----------
    df : pd.DataFrame
        Processed data with one row per participant
    verbose : bool
        Whether to show progress
    
    Returns
    -------
    pd.DataFrame
        Wide-format results with one row per participant
        Columns: participant_id, {condition}_{model}_ll, {condition}_{model}_alpha
    """
    # Create world
    world = create_world(n=N_EXPERIMENTS, m=M_PATIENTS)
    
    if verbose:
        print(f"World created: n={N_EXPERIMENTS}, m={M_PATIENTS}")
        print(f"  Utterances: {world.utterances}")
        print(f"  Possible outcomes: {len(world.possible_outcomes)}")
        print(f"\nFitting {len(MODEL_NAMES)} models × {len(GOAL_CONDITIONS)} conditions = {len(MODEL_NAMES) * len(GOAL_CONDITIONS)} fits per participant")
    
    # Process all participants
    all_results = []
    
    iterator = df.iterrows()
    if verbose:
        iterator = tqdm(list(iterator), desc="Fitting models")
    
    for idx, row in iterator:
        result = process_participant_wide(row, world)
        all_results.append(result)
    
    # Convert to DataFrame
    results_df = pd.DataFrame(all_results)
    
    # Reorder columns
    cols = ['participant_id']
    for condition in GOAL_CONDITIONS:
        cols.extend([f'{condition}_n_trials', f'{condition}_is_complete'])
        for model in MODEL_NAMES:
            cols.extend([f'{condition}_{model}_ll', f'{condition}_{model}_alpha'])
    
    # Only include columns that exist
    cols = [c for c in cols if c in results_df.columns]
    results_df = results_df[cols]
    
    return results_df

## Anonymization and data-dictionary generator

In [6]:
def create_anonymized_version(df):
    """
    Create anonymized version by removing identifying columns.
    Keeps participant_id as the only identifier.
    """
    # Columns to remove for anonymization
    id_columns = ['subject_id', 'prolific_pid', 'study_id', 'session_id', 
                  'start_time', 'completion_time', 'source_file']
    
    df_anon = df.copy()
    for col in id_columns:
        if col in df_anon.columns:
            df_anon = df_anon.drop(columns=[col])
    
    return df_anon

def profile_dataframe(df: pd.DataFrame, key_cols=None, max_levels=25) -> pd.DataFrame:
    """
    Build a "data dictionary" style profile of df:
    - dtype, missingness, unique counts
    - numeric stats (min/max/mean)
    - sample levels for low-cardinality columns
    """
    if key_cols is None:
        key_cols = []

    rows = []
    n = len(df)

    for col in df.columns:
        s = df[col]
        dtype = str(s.dtype)

        n_missing = int(s.isna().sum())
        pct_missing = (n_missing / n) if n else np.nan
        n_unique = int(s.nunique(dropna=True))

        info = {
            "column": col,
            "dtype": dtype,
            "n_rows": n,
            "n_missing": n_missing,
            "pct_missing": pct_missing,
            "n_unique": n_unique,
            "is_key": col in set(key_cols),
        }

        # Numeric summary
        if pd.api.types.is_numeric_dtype(s):
            info.update({
                "min": float(np.nanmin(s)) if n_missing < n else np.nan,
                "max": float(np.nanmax(s)) if n_missing < n else np.nan,
                "mean": float(np.nanmean(s)) if n_missing < n else np.nan,
                "example_values": ""
            })

        # Datetime-like (try parse)
        elif pd.api.types.is_datetime64_any_dtype(s):
            info.update({
                "min": str(s.min()) if n_missing < n else "",
                "max": str(s.max()) if n_missing < n else "",
                "mean": "",
                "example_values": ""
            })

        # Object / categorical: show levels if small
        else:
            info.update({"min": "", "max": "", "mean": ""})
            if n_unique <= max_levels:
                levels = s.dropna().astype(str).unique().tolist()
                info["example_values"] = "; ".join(levels[:max_levels])
            else:
                # show a few examples
                examples = s.dropna().astype(str).head(5).tolist()
                info["example_values"] = "; ".join(examples)

        rows.append(info)

    profile = pd.DataFrame(rows).sort_values(
        by=["is_key", "pct_missing", "n_unique"],
        ascending=[False, False, True]
    )

    # Key checks (if provided)
    for k in key_cols:
        if k in df.columns:
            dup_count = int(df[k].duplicated().sum())
            print(f"[KEY CHECK] {k}: unique={df[k].is_unique}, duplicates={dup_count}, missing={df[k].isna().sum()}")
        else:
            print(f"[KEY CHECK] {k}: column not found in df")

    return profile


## Run the full pipeline

Edit `INPUT_PATH` below if your processed-but-not-yet-fitted file lives
elsewhere.

In [7]:
# === EDIT THIS PATH IF YOUR INPUT IS ELSEWHERE ===
INPUT_PATH  = './raw_do_not_track/processed_speaker_n1_full.csv'  # produced by Stage A

# Output paths — defaults match the layout this directory ships with.
OUTPUT_FULL = './raw_do_not_track/speaker_n1_fitted.csv'           # full table (with identifiers)
OUTPUT_ANON = './speaker_n1_fitted_anonymized.csv'                 # public artifact
OUTPUT_DICT = './data_dictionary.csv'                              # public column profile

# 1. Load processed data (Stage A's output) and filter to completed participants.
df_raw = pd.read_csv(INPUT_PATH)
print(f"Loaded {len(df_raw)} participants from {INPUT_PATH}")
df = df_raw[df_raw['completion_status'] == 'completed'].copy()
print(f"Filtering to completed: {len(df_raw)} -> {len(df)}")

# 2. Fit all 7 models for each of the 3 scenarios per participant.
print(f"\n{'='*70}\nFITTING MODELS\n{'='*70}")
fitted_result = fit_all_participants_wide(df, verbose=True)

# 3. Merge fits back onto the participant table (1:1 by participant_id).
key = "participant_id"
overlap = df.columns.intersection(fitted_result.columns).difference([key])
df_fitted = df.merge(
    fitted_result.drop(columns=overlap),
    on=key, how="left", validate="one_to_one", indicator=True,
)
assert (df_fitted['_merge'] == 'both').all(), "merge produced unmatched rows"

# 4. Save the full version (still contains identifiers).
df_fitted.to_csv(OUTPUT_FULL, index=False)
print(f"\nSaved full fitted data to: {OUTPUT_FULL}")
print(f"  Rows: {len(df_fitted)}  Columns: {len(df_fitted.columns)}")

# 5. Anonymize and save the public version.
df_fitted_anon = create_anonymized_version(df_fitted)
df_fitted_anon.to_csv(OUTPUT_ANON, index=False)
print(f"Saved anonymized fitted data to: {OUTPUT_ANON}")
print(f"  Rows: {len(df_fitted_anon)}  Columns: {len(df_fitted_anon.columns)}")

# 6. Generate and save the data dictionary on the anonymized version.
dictionary = profile_dataframe(df_fitted_anon, key_cols=['participant_id'])
dictionary.to_csv(OUTPUT_DICT, index=False)
print(f"Saved data dictionary to: {OUTPUT_DICT}")
print(f"  Rows (columns documented): {len(dictionary)}")


FileNotFoundError: [Errno 2] No such file or directory: './raw_do_not_track/processed_speaker_n1_full.csv'